# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [35]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start,
        'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code in RETRY_STATUSES:
                if attempt < tries - 1:
                    time.sleep(2 ** attempt)
                    continue
                raise RuntimeError(
                    f"Weather API failed with status {r.status_code}"
                )

            r.raise_for_status()
            return pd.DataFrame(r.json()['daily'])

        except requests.HTTPError:
            raise

        except requests.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 ** attempt)

    raise RuntimeError("Weather API failed after all retries")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [36]:
try:
    wx = get_weather("2026-09-05", "2026-10-03")
except requests.RequestException as e:
    print("Weather API request failed:", e)
    wx = pd.DataFrame()
except RuntimeError as e:
    print("Weather API retries exhausted:", e)
    wx = pd.DataFrame()

used_fallback = False

print(wx)

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
1   2026-09-06                27.2                1.6
2   2026-09-07                27.7                0.0
3   2026-09-08                29.4                0.0
4   2026-09-09                30.9                0.0
5   2026-09-10                34.2                3.2
6   2026-09-11                28.0               10.4
7   2026-09-12                27.5               21.5
8   2026-09-13                31.3                5.8
9   2026-09-14                28.7                0.3
10  2026-09-15                25.8                0.0
11  2026-09-16                29.7                0.0
12  2026-09-17                33.7                0.6
13  2026-09-18                30.0                1.6
14  2026-09-19                27.1                2.3
15  2026-09-20                30.5                0.1
16  2026-09-21                24.7                1.0
17  2026-09-22              

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [37]:
FALLBACK = pd.DataFrame({
    "time": ["2026-09-05", "2026-10-03"],
    "temperature_2m_max": [30.1, 18.4],
    "precipitation_sum": [0.0, 22.5],
})
if wx.empty:
    used_fallback = True
    wx = FALLBACK.copy()

wx["source"] = "sample" if used_fallback else "open-meteo"

print(wx.head())

         time  temperature_2m_max  precipitation_sum      source
0  2026-09-05                29.4                2.5  open-meteo
1  2026-09-06                27.2                1.6  open-meteo
2  2026-09-07                27.7                0.0  open-meteo
3  2026-09-08                29.4                0.0  open-meteo
4  2026-09-09                30.9                0.0  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [38]:
wx = wx.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm"
})

wx["date"] = pd.to_datetime(wx["date"])

print(wx.head())

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-09-06      27.2        1.6  open-meteo
2 2026-09-07      27.7        0.0  open-meteo
3 2026-09-08      29.4        0.0  open-meteo
4 2026-09-09      30.9        0.0  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [39]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) >= 2
assert wx['date'].dtype == 'datetime64[ns]'
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'

print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [40]:
rain_sep5 = wx.loc[wx["date"] == "2026-09-05", "precip_mm"].iloc[0]
rain_oct3 = wx.loc[wx["date"] == "2026-10-03", "precip_mm"].iloc[0]

difference = abs(rain_oct3 - rain_sep5)

wetter = "October 3" if rain_oct3 > rain_sep5 else "September 5"

print(f"{wetter} was wetter by {difference:.1f} mm of precipitation.")

October 3 was wetter by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [41]:
from datetime import datetime

print("Source: Open-Meteo Historical Weather API")
print("Endpoint:", BASE)
print("Coordinates: 38.03, -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Retrieved:", datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Data type:", "Fallback sample" if used_fallback else "Real API data")

Source: Open-Meteo Historical Weather API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Retrieved: 2026-10-11 22:04
Data type: Real API data


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

In case there is an empty DataFrame when get_weather() does not retrieve any data due to failure of the API call, then the weather data will not be available for merging with sales data of 200 days. The chart will either have blank rainfall data or no data for those days, thus making the connection between rainfall and revenue false. This problem can be overlooked since there are no errors in the code.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [42]:
# Get Orlando weather
orlando = get_weather(
    "2026-09-05", "2026-10-03",
    lat=28.54, lon=-81.38
)

# Clean Orlando data
orlando = orlando.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm"
})

orlando["date"] = pd.to_datetime(orlando["date"])
orlando["source"] = "open-meteo"

# Keep only the two dates
dates = pd.to_datetime(["2026-09-05", "2026-10-03"])

charlottesville = wx[wx["date"].isin(dates)].copy()
charlottesville["city"] = "Charlottesville"

orlando = orlando[orlando["date"].isin(dates)].copy()
orlando["city"] = "Orlando"

# Combine both cities
combined = pd.concat([charlottesville, orlando], ignore_index=True)

print(combined)

        date  temp_max  precip_mm      source             city
0 2026-09-05      29.4        2.5  open-meteo  Charlottesville
1 2026-10-03      22.0       14.7  open-meteo  Charlottesville
2 2026-09-05      32.1        3.9  open-meteo          Orlando
3 2026-10-03      30.4       14.5  open-meteo          Orlando
